In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
df_churn = pd.read_csv('course_lead_scoring_2026.csv')
df_churn.head(2)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1


<font size="4"> <b>Check if the missing values are presented in the features.<br>
If there are missing values:</b><br>
1. For categorical features, replace them with 'NA'<br>
2. For numerical features, replace with with 0.0 </font><br>

In [5]:
df_churn.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df_churn.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
categorical_cols = df_churn.select_dtypes(
    include=["object", "string", "category"]
).columns

numerical_cols = df_churn.select_dtypes(include="number").columns

In [8]:
for col in categorical_cols:
    if isinstance(df_churn[col].dtype, pd.CategoricalDtype):
        if "NA" not in df_churn[col].cat.categories:
            df_churn[col] = df_churn[col].cat.add_categories(["NA"])
    df_churn[col] = df_churn[col].fillna("NA")

df_churn[numerical_cols] = df_churn[numerical_cols].fillna(0.0)

df_churn.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

<font size="5"><b><u> Question 1. What is the most frequent observation (mode) for the column industry? </u></b></font><br>

In [9]:
df_churn["industry"].mode()

0    technology
Name: industry, dtype: object

<font size="5"><b><u> Question 2. Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features. </u></b></font><br>

In [12]:
df_churn[numerical_cols].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


<font size="5"><b><u> Question 3. Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only. Round the scores to 2 decimals using round(score, 2). </u></b></font><br>

In [15]:
df_full_train, df_test = train_test_split(df_churn, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [16]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

In [18]:
mutual_info_score(df_full_train.industry, df_full_train.converted)

np.float64(0.0028984818739526547)

In [30]:
categorical_cols = df_train.select_dtypes(
    include=["object", "string", "category"]
).columns

numerical_cols = df_train.select_dtypes(include="number").columns

In [26]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

df_full_train[categorical_cols].apply(mutual_info_converted_score)

lead_source          0.034761
industry             0.002898
employment_status    0.020313
location             0.001050
dtype: float64

In [27]:
mi = df_full_train[categorical_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

<font size="5"><b><u> Question 4. Fit the model on the training dataset<br>
Calculate the accuracy on the validation dataset and round it to 2 decimal digits</u></b></font><br>

In [44]:
features = list(categorical_cols) + list(numerical_cols)

train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [45]:
model = LogisticRegression(
    solver="liblinear", C=1.0, max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [46]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
accuracy = (y_val == converted_decision).mean()
print(round(accuracy, 2))

0.64


<font size="5"><b><u> Question 5. </u></b></font><br> 
Train a model using the same features and parameters as in Q4 (without rounding).<br> 
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.<br>
For each feature, calculate the difference between the original accuracy and the accuracy without the feature. 

In [42]:
def train_and_score(selected_features):
    train_dicts = df_train[selected_features].to_dict(orient="records")
    val_dicts = df_val[selected_features].to_dict(orient="records")

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)

    return (y_val == converted_decision).mean()

In [43]:
original_accuracy = train_and_score(features)
print("Original accuracy:", original_accuracy)

results = []

for feature in features:
    remaining_features = [f for f in features if f != feature]

    accuracy_without = train_and_score(remaining_features)
    difference = original_accuracy - accuracy_without

    results.append({
        "excluded_feature": feature,
        "accuracy_without": accuracy_without,
        "difference": difference
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("difference").reset_index(drop=True)

display(results_df)

Original accuracy: 0.645


,excluded_feature,accuracy_without,difference
0,annual_income,0.724,-0.079
1,industry,0.645,0.000
2,location,0.645,0.000
3,employment_status,0.645,0.000
4,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
6,lead_source,0.642,0.003
7,interaction_count,0.601,0.044


<font size="5"><b><u> Question 6. train a regularized logistic regression. </u></b></font><br>
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001]. <br>
Train models using all the features as in Q4.<br>
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

In [47]:
results = []

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(
        solver="liblinear",
        C=c,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)

    accuracy = (y_val == converted_decision).mean()

    results.append({
        "C": c,
        "validation_accuracy": round(accuracy, 3)
    })

results_df = pd.DataFrame(results)
display(results_df.style.format({
    "C": "{:.6f}",
    "validation_accuracy": "{:.3f}"
}))

,C,validation_accuracy
0,0.000001,0.598
1,0.000010,0.598
2,0.000100,0.613
3,0.001000,0.645
